# CharXiv Modeling, Baselines (Checkpoint 4)

These are trivial and simple reference models for predicting failure (`y = 1` means incorrect) on the
CharXiv reasoning task, fit separately for each of the three target models, GPT-4o, Claude-3-5-Sonnet,
and GPT-4o-Random. They are the reference points that every tuned family in
`17_charxiv_modeling_experiments.ipynb` must beat.

Everything runs through the shared, per-fold-fitted evaluation framework from Checkpoint 3
(`src/eval/core.py` and `src/splits/splitters.py`) with the Checkpoint 2 default preprocessing
(`src.features.preprocessing.build_preprocessor`, the eleven kept DeLeAn demand dimensions with item
metadata off). The primary key performance indicator is ROC-AUC (see `kpis.md`), and the split
rationale and leakage taxonomy are in `evaluation_plan.md`.

GPT-4o-Random is a random-answer control that fails about 90 percent of items. It stays near chance on
the demand dimensions, because reasoning demand does not predict a random answer. The 200-item test set
stays sealed and is never read here.

## 1. Setup and data

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src.features.preprocessing as P
from src.splits.splitters import ItemHoldoutSplit
from src.eval.core import evaluate, METRIC_COLS
from src.models import registry

RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
splitter = ItemHoldoutSplit(5, 3)                       # 15 paper-grouped folds
TARGETS = P.TARGET_MODELS

con = P.connect(); train_ids, test_ids = P.get_split(con)
df = P.make_design_matrix(con, train_ids)               # one row per item
con.close()

def for_target(t):
    d = df.copy(); d["y"] = d[P.ycol(t)].to_numpy(); return d

print("items:", len(df))
display(df[P.TARGET_YCOLS].mean().round(3).rename("failure_rate").to_frame())
print("baseline families:", registry.BASELINE_FAMILIES)

items: 800


,failure_rate
y__GPT-4o,0.514
y__Claude-3-5-Sonnet,0.390
y__GPT-4o-Random,0.899


baseline families: ['dummy_most_frequent', 'dummy_stratified', 'logistic', 'random_forest']


## 2. Baseline cross-validation, per target

Each family is evaluated for each target with the same 15 paper-grouped folds. We report the
key-performance-indicator set (ROC-AUC primary, then balanced accuracy and F1) as a mean over folds,
with the ROC-AUC cross-fold standard deviation.

In [2]:
def run_family(target, family):
    make_pre = lambda: P.build_preprocessor()
    summary, _, _ = evaluate(for_target(target), make_pre, registry.FACTORIES[family], splitter)
    row = {"target": target, "family": family, "config": "baseline"}
    row.update({c: round(float(summary[c]), 4) for c in METRIC_COLS})
    row["roc_auc_sd"] = round(float(summary["roc_auc_sd"]), 4)
    return row

base = pd.DataFrame([run_family(t, fam) for t in TARGETS for fam in registry.BASELINE_FAMILIES])
base.to_csv(RESULTS / "model_comparison_baselines.csv", index=False)
print("wrote results/model_comparison_baselines.csv")

wrote results/model_comparison_baselines.csv


## 3. Baseline results

In [3]:
cols = ["target", "family", "roc_auc", "roc_auc_sd", "bal_acc", "f1"]
tbl = base[cols].copy()
display(tbl.set_index(["target", "family"]))
# ROC-AUC by family x target for a quick read
display(base.pivot_table(index="family", columns="target", values="roc_auc").reindex(
    registry.BASELINE_FAMILIES)[TARGETS].round(4))

roc_auc  roc_auc_sd  bal_acc      f1
target            family                                                   
GPT-4o            dummy_most_frequent   0.5000      0.0000   0.5000  0.6788
                  dummy_stratified      0.5256      0.0418   0.5256  0.5130
                  logistic              0.6758      0.0336   0.6304  0.6488
                  random_forest         0.6584      0.0381   0.6310  0.6526
Claude-3-5-Sonnet dummy_most_frequent   0.5000      0.0000   0.5000  0.0000
                  dummy_stratified      0.4936      0.0365   0.4936  0.3872
                  logistic              0.6275      0.0490   0.5680  0.3896
                  random_forest         0.6224      0.0499   0.5611  0.3769
GPT-4o-Random     dummy_most_frequent   0.5000      0.0000   0.5000  0.9467
                  dummy_stratified      0.5090      0.0276   0.5090  0.9073
                  logistic              0.5799      0.0528   0.5000  0.9467
                  random_forest         0.6090      0.0524   0.5000  0.9467

target,GPT-4o,Claude-3-5-Sonnet,GPT-4o-Random
family,,,
dummy_most_frequent,0.5000,0.5000,0.5000
dummy_stratified,0.5256,0.4936,0.5090
logistic,0.6758,0.6275,0.5799
random_forest,0.6584,0.6224,0.6090


## 4. Learnability decision rule

A learner is credited with real signal only if its ROC-AUC beats the stronger Dummy baseline by more
than one cross-fold standard deviation (the rule stated in `kpis.md`), evaluated per target. The two
real models clear the trivial floor; the random control does not on the demand dimensions, which is
the expected negative-control result.

In [4]:
for t in TARGETS:
    sub = base[base.target == t]
    dummy_auc = sub.loc[sub.family.str.startswith("dummy"), "roc_auc"].max()
    print(f"=== {t} (Dummy reference ROC-AUC = {dummy_auc:.4f}) ===")
    for _, r in sub.sort_values("roc_auc", ascending=False).iterrows():
        learnable = r.roc_auc - r.roc_auc_sd > dummy_auc
        print(f"  {r.family:20s} ROC-AUC {r.roc_auc:.4f} ± {r.roc_auc_sd:.4f}   beats Dummy+1SD: {learnable}")

=== GPT-4o (Dummy reference ROC-AUC = 0.5256) ===
  logistic             ROC-AUC 0.6758 ± 0.0336   beats Dummy+1SD: True
  random_forest        ROC-AUC 0.6584 ± 0.0381   beats Dummy+1SD: True
  dummy_stratified     ROC-AUC 0.5256 ± 0.0418   beats Dummy+1SD: False
  dummy_most_frequent  ROC-AUC 0.5000 ± 0.0000   beats Dummy+1SD: False
=== Claude-3-5-Sonnet (Dummy reference ROC-AUC = 0.5000) ===
  logistic             ROC-AUC 0.6275 ± 0.0490   beats Dummy+1SD: True
  random_forest        ROC-AUC 0.6224 ± 0.0499   beats Dummy+1SD: True
  dummy_most_frequent  ROC-AUC 0.5000 ± 0.0000   beats Dummy+1SD: False
  dummy_stratified     ROC-AUC 0.4936 ± 0.0365   beats Dummy+1SD: False
=== GPT-4o-Random (Dummy reference ROC-AUC = 0.5090) ===
  random_forest        ROC-AUC 0.6090 ± 0.0524   beats Dummy+1SD: True
  logistic             ROC-AUC 0.5799 ± 0.0528   beats Dummy+1SD: True
  dummy_stratified     ROC-AUC 0.5090 ± 0.0276   beats Dummy+1SD: False
  dummy_most_frequent  ROC-AUC 0.5000 ± 0.0000

## 6. Reproducibility check and takeaway

The per-target Logistic Regression and Random Forest numbers here reproduce the Checkpoint 3 figures
in `results/cv_metrics__item_holdout__2026-07-03.csv`, confirming the modeling code sits on the same
evaluation framework. For the two real models the default Logistic Regression is the model to beat.
The tuned families and the interpretability, robustness, and model-selection work continue in
`17_charxiv_modeling_experiments.ipynb`.

In [5]:
ckpt3 = pd.read_csv(RESULTS / "cv_metrics__item_holdout__2026-07-03.csv")
name_map = {"logistic": "LogisticRegression", "random_forest": "RandomForest"}
rows = []
for t in TARGETS:
    for fam, learner in name_map.items():
        c3 = ckpt3[(ckpt3.target == t) & (ckpt3.learner == learner)]["roc_auc"]
        here = base[(base.target == t) & (base.family == fam)]["roc_auc"]
        rows.append({"target": t, "family": fam,
                     "checkpoint3": float(c3.iloc[0]) if len(c3) else np.nan,
                     "here": float(here.iloc[0]) if len(here) else np.nan})
display(pd.DataFrame(rows).set_index(["target", "family"]).round(4))
assert set(df.item_id).isdisjoint(set(test_ids)), "test items leaked into the design matrix!"
print("OK: the 200 sealed test ids never enter any training or evaluation matrix.")

checkpoint3    here
target            family                            
GPT-4o            logistic            0.6758  0.6758
                  random_forest       0.6584  0.6584
Claude-3-5-Sonnet logistic            0.6275  0.6275
                  random_forest       0.6224  0.6224
GPT-4o-Random     logistic            0.5799  0.5799
                  random_forest       0.6090  0.6090

OK: the 200 sealed test ids never enter any training or evaluation matrix.
